# retina-08-compression

**Make it deployable, then prove it still works.**

The model has to run on the laptop already inside the screening van — no network, CPU
only. This notebook shrinks it and re-tests everything that compression can quietly
break.

---

## BEFORE YOU RUN — inputs this notebook needs

| # | Artefact | Slug | What is taken from it |
|---|---|---|---|
| 1 | Multi-head model | `retina-multihead` | `retina_multihead.keras` — the 18.4 MB original |
| 2 | External datasets | `retina-external-512` | 4,556 images to re-test on |
| 3 | Week 11 predictions | `retina-generalisation` | `predictions_external.csv` — the reference to compare against |

### Settings

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **either — GPU not needed** | see below |
| **Internet** | **OFF** | nothing is downloaded |

**This is the first notebook in the project that needs no GPU.** The Keras model is
loaded only so it can be *converted* (a graph operation on CPU); it is never asked for a
prediction, because the reference predictions come from Week 11's saved CSV. And a
TFLite interpreter always runs on CPU, GPU present or not.

So: if the GPU is already on, leave it — it costs you nothing and saves a restart. If
you are near your weekly GPU quota, switch to CPU and lose nothing. Cell 1 prints which
you have and does not assert either way.

Small inputs again — this attaches in seconds.

### Roughly how long

| Step | Time |
|---|---|
| Cells 1–4 — convert and verify four variants | 3–5 min |
| Cell 5 — latency benchmark | 2 min |
| Cell 6 — agreement on 4,556 images × 4 variants | 8–12 min |
| Cells 7–9 — safety, pathology, decision | 1 min |

**Run top to bottom. No edits needed.**

> **One thing deliberately NOT re-tested.** The plan was to re-check Grad-CAM
> localisation after compression. Week 10 and Week 11 both measured it as flat at the
> chance level, so there is nothing left to degrade — and TFLite has no autodiff, so the
> measurement is not even constructible. Cell 9 says so rather than quietly skipping it.

In [1]:
# ============================================================================
# CELL 1 - Setup
# ============================================================================
import datetime, json, os, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from scipy.stats import spearmanr
from tqdm.auto import tqdm

AUTOTUNE = tf.data.AUTOTUNE
SEED = 42
SIZE = 224
DEFECTS = ["underexposed", "defocus", "obstruction", "off_centre"]
CLASSES = ["Good", "Usable", "Reject"]
GOOD, USABLE, REJECT = 0, 1, 2
keras.utils.set_random_seed(SEED)


def resolve(slug, root=Path("/kaggle/input"), max_depth=3, required=True):
    for depth in range(1, max_depth + 1):
        hits = [p for p in root.glob("/".join(["*"] * (depth - 1) + [slug])) if p.is_dir()]
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(f"'{slug}' is not attached - Add Data in the right-hand panel")
    return None


def find_file(name, root=Path("/kaggle/input"), required=True):
    hits = sorted(root.rglob(name))
    if not hits:
        if required:
            raise FileNotFoundError(f"'{name}' not found under /kaggle/input")
        return None
    return hits[0]


print("resolving inputs ...\n")
CKPT      = find_file("retina_multihead.keras")
EXTERNAL  = resolve("retina-external-512")
EXT_IMAGES = EXTERNAL / "external_512"
REF_CSV   = find_file("predictions_external.csv")

for label, p in [("keras model", CKPT), ("external images", EXT_IMAGES),
                 ("week-11 predictions", REF_CSV)]:
    assert p is not None and p.exists(), f"missing: {label}"
    print(f"  OK  {label:22s} {p}")

WORK   = Path("/kaggle/working")
MODELS = WORK / "tflite"
RESULTS = WORK / "results"
for d in (MODELS, RESULTS):
    d.mkdir(exist_ok=True)

print(f"\nTensorFlow {tf.__version__} | Keras {keras.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU') or 'none (fine - TFLite runs on CPU)'}")


def record(name, params, metrics, notes=""):
    payload = {"name": name,
               "when": datetime.datetime.now().isoformat(timespec="seconds"),
               "params": params, "metrics": metrics, "notes": notes}
    (RESULTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str))
    print(f"recorded {name}")
    return payload

resolving inputs ...

  OK  keras model            /kaggle/input/datasets/izuchukwuoruche/retina-multihead/retina_multihead.keras
  OK  external images        /kaggle/input/datasets/izuchukwuoruche/retina-external-512/external_512
  OK  week-11 predictions    /kaggle/input/datasets/izuchukwuoruche/retina-generalisation/predictions_external.csv

TensorFlow 2.20.0 | Keras 3.13.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# ============================================================================
# CELL 2 - The original, and the reference it must agree with
# ============================================================================
mh = keras.models.load_model(CKPT, compile=False)
KERAS_MB = os.path.getsize(CKPT) / 1e6
print(f"retina_multihead.keras   {KERAS_MB:.2f} MB")

ref = pd.read_csv(REF_CSV)
print(f"\n{len(ref):,} reference predictions from Week 11")
print(ref.dataset.value_counts().to_string())

# Week 11 already ran every external image through the uncompressed model, so its
# predictions ARE the reference. Nothing is re-predicted on GPU here - the whole
# point is to compare a compressed CPU model against that frozen baseline.
need = ["filename", "dataset", "quality", "pred", "p_good", "p_usable", "p_reject"] \
       + [f"sev_{d}" for d in DEFECTS]
missing = [c for c in need if c not in ref.columns]
assert not missing, f"reference csv is missing {missing} - is this Week 11's output?"

ref_grad = ref[["p_good", "p_usable", "p_reject"]].values.astype("float32")
ref_sev  = ref[[f"sev_{d}" for d in DEFECTS]].values.astype("float32")
print(f"\nreference arrays: gradability {ref_grad.shape}  defects {ref_sev.shape}")


def decode_np(fn, size=SIZE):
    """Same preprocessing as every earlier notebook. float32 in [0, 255]."""
    img = tf.io.decode_png(tf.io.read_file(str(EXT_IMAGES / fn)), channels=3)
    img = tf.image.resize(img, (size, size), method="area")
    return tf.cast(img, tf.float32).numpy()


print("\nloading images into memory (one pass, reused by every variant) ...")
X = np.stack([decode_np(f) for f in tqdm(ref.filename, desc="decode")])
print(f"X: {X.shape}  {X.nbytes / 1e9:.2f} GB")

I0000 00:00:1791930063.396450      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791930063.400357      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


retina_multihead.keras   18.37 MB

4,556 reference predictions from Week 11
dataset
fqs         2246
deepdrid    1600
idrid        516
drimdb       194

reference arrays: gradability (4556, 3)  defects (4556, 4)

loading images into memory (one pass, reused by every variant) ...


decode:   0%|          | 0/4556 [00:00<?, ?it/s]

X: (4556, 224, 224, 3)  2.74 GB


In [3]:
# ============================================================================
# CELL 3 - Convert to four compression levels
# ============================================================================
# int8 calibration MUST see real images. Random noise produces wrong activation
# ranges and a model that is quietly inaccurate rather than broken.
rng = np.random.default_rng(SEED)
cal_idx = rng.choice(len(X), size=min(200, len(X)), replace=False)
print(f"int8 calibration set: {len(cal_idx)} real images")


def representative_dataset():
    for i in cal_idx:
        yield [X[i:i + 1]]


def convert(name, tune):
    t0 = time.time()
    c = tf.lite.TFLiteConverter.from_keras_model(mh)
    tune(c)
    blob = c.convert()
    path = MODELS / f"retina_multihead_{name}.tflite"
    path.write_bytes(blob)
    mb = len(blob) / 1e6
    print(f"  {name:9s} {mb:6.2f} MB   ({KERAS_MB / mb:.1f}x smaller)   "
          f"converted in {time.time() - t0:.0f}s")
    return mb


print("\nconverting ...")
sizes = {}
sizes["float32"] = convert("float32", lambda c: None)
sizes["float16"] = convert("float16", lambda c: (
    setattr(c, "optimizations", [tf.lite.Optimize.DEFAULT]),
    setattr(c, "target_spec", tf.lite.TargetSpec(supported_types=[tf.float16]))))
sizes["dynamic"] = convert("dynamic", lambda c:
    setattr(c, "optimizations", [tf.lite.Optimize.DEFAULT]))
try:
    sizes["int8"] = convert("int8", lambda c: (
        setattr(c, "optimizations", [tf.lite.Optimize.DEFAULT]),
        setattr(c, "representative_dataset", representative_dataset)))
except Exception as e:
    print(f"  int8 conversion FAILED: {type(e).__name__}: {str(e)[:200]}")
    print("  continuing without it - the other three are enough to decide")

print(f"\n  {'keras':9s} {KERAS_MB:6.2f} MB   (the original)")
print("""
WHAT EACH ONE DOES
  float32  no quantisation - just the TFLite runtime. The floor for "did conversion
           itself change anything".
  float16  weights stored at half precision. Usually near-lossless.
  dynamic  int8 weights, float activations computed on the fly. Smallest, no
           calibration data needed.
  int8     weights AND activations quantised, calibrated on 200 real images.
           Fastest on CPU, and the only one that can actually change an answer.""")

int8 calibration set: 200 real images

converting ...
INFO:tensorflow:Assets written to: /tmp/tmpvq16c4kp/assets


INFO:tensorflow:Assets written to: /tmp/tmpvq16c4kp/assets


Saved artifact at '/tmp/tmpvq16c4kp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  Dict[['gradability', TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)], ['defects', TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)]]
Captures:
  138929125737488: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  138929125737296: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  138930709232720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930709233296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930709233104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930709226768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930709233488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930715850896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930707644880: TensorSpec(shape=(), dty

W0000 00:00:1791930154.480750      22 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1791930154.480780      22 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1791930154.689715      22 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


  float32    17.36 MB   (1.1x smaller)   converted in 16s
INFO:tensorflow:Assets written to: /tmp/tmp8o9pjv2w/assets


...

  dynamic     4.88 MB   (3.8x smaller)   converted in 16s
INFO:tensorflow:Assets written to: /tmp/tmpt8g1229t/assets


INFO:tensorflow:Assets written to: /tmp/tmpt8g1229t/assets


Saved artifact at '/tmp/tmpt8g1229t'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  Dict[['gradability', TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)], ['defects', TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)]]
Captures:
  138929125737488: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  138929125737296: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  138930709232720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930709233296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930709233104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930709226768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930709233488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930715850896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138930707644880: TensorSpec(shape=(), dty

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1791930201.778596      22 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1791930201.778626      22 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32


  int8        5.25 MB   (3.5x smaller)   converted in 77s

  keras      18.37 MB   (the original)

WHAT EACH ONE DOES
  float32  no quantisation - just the TFLite runtime. The floor for "did conversion
           itself change anything".
  float16  weights stored at half precision. Usually near-lossless.
  dynamic  int8 weights, float activations computed on the fly. Smallest, no
           calibration data needed.
  int8     weights AND activations quantised, calibrated on 200 real images.
           Fastest on CPU, and the only one that can actually change an answer.


In [4]:
# ============================================================================
# CELL 4 - Identify the outputs BY SHAPE, not by order
# ============================================================================
# The same trap as cell 8 of Week 10, one layer down. Keras sorted the output dict
# alphabetically; TFLite then assigns opaque names like 'StatefulPartitionedCall:0'
# and gives no guarantee which is which. Reading them positionally would silently
# swap gradability for defects - and every shape downstream would still check out.
#
# The defect head has 4 units and the gradability head has 3, so SHAPE identifies
# them unambiguously. That is what we key on.
def open_interpreter(name):
    it = tf.lite.Interpreter(model_path=str(MODELS / f"retina_multihead_{name}.tflite"))
    it.resize_tensor_input(it.get_input_details()[0]["index"], [1, SIZE, SIZE, 3])
    it.allocate_tensors()
    inp = it.get_input_details()[0]["index"]
    by_units = {int(d["shape"][-1]): d["index"] for d in it.get_output_details()}
    assert 3 in by_units and len(DEFECTS) in by_units, (
        f"cannot identify outputs by shape: {by_units}. "
        f"Expected one output with 3 units and one with {len(DEFECTS)}.")
    return it, inp, by_units[3], by_units[len(DEFECTS)]


for name in sizes:
    it = tf.lite.Interpreter(model_path=str(MODELS / f"retina_multihead_{name}.tflite"))
    it.allocate_tensors()
    print(f"\n{name}:")
    for d in it.get_output_details():
        units = int(d["shape"][-1])
        print(f"   {str(tuple(int(v) for v in d['shape'])):12s} {units} units "
              f"-> {'defects' if units == len(DEFECTS) else 'gradability'}   "
              f"name {d['name']!r}")
    _ = open_interpreter(name)      # asserts
print("\nall variants: outputs identified by shape, assertions passed")


float32:
   (1, 4)       4 units -> defects   name 'StatefulPartitionedCall_1:0'
   (1, 3)       3 units -> gradability   name 'StatefulPartitionedCall_1:1'

float16:
   (1, 4)       4 units -> defects   name 'StatefulPartitionedCall_1:0'
   (1, 3)       3 units -> gradability   name 'StatefulPartitionedCall_1:1'

dynamic:
   (1, 4)       4 units -> defects   name 'StatefulPartitionedCall_1:0'
   (1, 3)       3 units -> gradability   name 'StatefulPartitionedCall_1:1'

int8:
   (1, 4)       4 units -> defects   name 'StatefulPartitionedCall_1:0'
   (1, 3)       3 units -> gradability   name 'StatefulPartitionedCall_1:1'

all variants: outputs identified by shape, assertions passed


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
# ============================================================================
# CELL 5 - Size and CPU latency
# ============================================================================
# Batch 1 on purpose. In the van the operator photographs one eye and waits for one
# answer - average throughput over a big batch is not the number that matters.
WARMUP, TIMED = 10, 60
bench = {}
probe = X[:WARMUP + TIMED]

for name in sizes:
    it, inp, gi, di = open_interpreter(name)
    for j in range(WARMUP):                      # let XNNPACK settle
        it.set_tensor(inp, probe[j:j + 1]); it.invoke()
    t0 = time.perf_counter()
    for j in range(WARMUP, WARMUP + TIMED):
        it.set_tensor(inp, probe[j:j + 1]); it.invoke()
    ms = (time.perf_counter() - t0) / TIMED * 1000
    bench[name] = {"size_mb": sizes[name], "ms_per_image": ms}
    print(f"  {name:9s} {sizes[name]:6.2f} MB   {ms:7.1f} ms/image   "
          f"{1000 / ms:5.1f} images/s")

print("""
READ THIS CAREFULLY - smaller is not automatically faster
  float16 and dynamic-range quantisation shrink the FILE, but on a CPU without
  native half-precision the weights are dequantised on the fly, and dynamic range
  adds a quantise/dequantise step around each op. Either can end up SLOWER than
  plain float32 while being half the size.

  Only full int8 moves both numbers the right way, because the arithmetic itself
  is integer.

  These timings are from this Kaggle CPU. The van laptop will differ, so treat the
  RANKING as the result and the absolute milliseconds as indicative.""")

  float32    17.36 MB      17.8 ms/image    56.3 images/s
  float16     8.74 MB      18.1 ms/image    55.2 images/s
  dynamic     4.88 MB      32.6 ms/image    30.7 images/s
  int8        5.25 MB      18.7 ms/image    53.5 images/s

READ THIS CAREFULLY - smaller is not automatically faster
  float16 and dynamic-range quantisation shrink the FILE, but on a CPU without
  native half-precision the weights are dequantised on the fly, and dynamic range
  adds a quantise/dequantise step around each op. Either can end up SLOWER than
  plain float32 while being half the size.

  Only full int8 moves both numbers the right way, because the arithmetic itself
  is integer.

  These timings are from this Kaggle CPU. The van laptop will differ, so treat the
  RANKING as the result and the absolute milliseconds as indicative.


In [6]:
# ============================================================================
# CELL 6 - Does it still give the same answers?
# ============================================================================
# Every external image, every variant, against Week 11's frozen predictions.
agree = {}
preds = {}
for name in sizes:
    it, inp, gi, di = open_interpreter(name)
    G = np.empty((len(X), 3), "float32")
    D = np.empty((len(X), len(DEFECTS)), "float32")
    for j in tqdm(range(len(X)), desc=f"{name:9s}"):
        it.set_tensor(inp, X[j:j + 1]); it.invoke()
        G[j] = it.get_tensor(gi)[0]
        D[j] = it.get_tensor(di)[0]
    preds[name] = (G, D)

    same = (G.argmax(1) == ref_grad.argmax(1))
    # a flipped argmax only matters when the original was CONFIDENT. Near a 3-way
    # tie, a 1e-7 nudge flips the label and means nothing.
    conf = ref_grad.max(1) > 0.60
    agree[name] = {
        "grad_max_abs_diff": float(np.abs(G - ref_grad).max()),
        "defect_max_abs_diff": float(np.abs(D - ref_sev).max()),
        "argmax_agreement": float(same.mean()),
        "argmax_agreement_confident": float(same[conf].mean()),
        "n_confident": int(conf.sum()),
        "defect_mae": float(np.abs(D - ref_sev).mean())}
    a = agree[name]
    print(f"  {name:9s} grad maxdiff {a['grad_max_abs_diff']:.2e}  "
          f"defect MAE {a['defect_mae']:.2e}  "
          f"argmax {a['argmax_agreement']:.1%} (confident {a['argmax_agreement_confident']:.1%})")

print(f"""
  'confident' means the original model put more than 0.60 on its top class
  ({agree[list(sizes)[0]]['n_confident']:,} of {len(X):,} images). Those are the predictions a
  disagreement would actually matter for. On the rest the model was already
  undecided, and a flip there is numerical noise, not a behaviour change.""")

float32  :   0%|          | 0/4556 [00:00<?, ?it/s]

  float32   grad maxdiff 5.44e-05  defect MAE 1.45e-06  argmax 100.0% (confident 100.0%)


float16  :   0%|          | 0/4556 [00:00<?, ?it/s]

  float16   grad maxdiff 2.73e-02  defect MAE 1.00e-03  argmax 99.6% (confident 100.0%)


dynamic  :   0%|          | 0/4556 [00:00<?, ?it/s]

  dynamic   grad maxdiff 7.05e-01  defect MAE 2.69e-02  argmax 90.5% (confident 94.9%)


int8     :   0%|          | 0/4556 [00:00<?, ?it/s]

  int8      grad maxdiff 9.71e-01  defect MAE 1.01e-01  argmax 58.8% (confident 60.2%)

  'confident' means the original model put more than 0.60 on its top class
  (3,899 of 4,556 images). Those are the predictions a
  disagreement would actually matter for. On the rest the model was already
  undecided, and a flip there is numerical noise, not a behaviour change.


In [7]:
# ============================================================================
# CELL 7 - Does the SAFETY metric survive compression?
# ============================================================================
# The one that decides deployment: of the photographs a human called ungradable,
# how many does each variant wave through as Good?
safety = {}
rows = []
for name in sizes:
    G, _ = preds[name]
    p = G.argmax(1)
    per = {}
    for ds in sorted(ref.dataset.unique()):
        m = (ref.dataset == ds).values & (ref.quality == REJECT).values
        if m.sum() == 0:
            continue
        per[ds] = float((p[m] == GOOD).mean())
    safety[name] = per
    rows.append({"variant": name, **{k: v for k, v in per.items()}})

# and the uncompressed reference, computed the same way
per = {}
for ds in sorted(ref.dataset.unique()):
    m = (ref.dataset == ds).values & (ref.quality == REJECT).values
    if m.sum():
        per[ds] = float((ref_grad.argmax(1)[m] == GOOD).mean())
rows.insert(0, {"variant": "keras (uncompressed)", **per})

tab = pd.DataFrame(rows).set_index("variant")
print("Reject-as-Good rate by dataset and variant\n")
print((tab * 100).round(2).to_string())
print("""
  Compare each row against 'keras (uncompressed)'. A variant that holds these
  rates has preserved the property the whole project exists for. A variant that
  raises them has traded safety for megabytes, and the trade is not acceptable
  at any size.""")

record("w12-safety-under-compression",
       {"n_images": int(len(X))},
       {"reject_as_good": tab.to_dict("index")})

Reject-as-Good rate by dataset and variant

                      deepdrid  drimdb   fqs
variant                                     
keras (uncompressed)      8.55     0.0  2.13
float32                   8.55     0.0  2.13
float16                   8.43     0.0  2.03
dynamic                   9.14     0.0  2.03
int8                      3.33     0.0  0.21

  Compare each row against 'keras (uncompressed)'. A variant that holds these
  rates has preserved the property the whole project exists for. A variant that
  raises them has traded safety for megabytes, and the trade is not acceptable
  at any size.
recorded w12-safety-under-compression


{'name': 'w12-safety-under-compression',
 'when': '2026-10-13T22:31:09',
 'params': {'n_images': 4556},
 'metrics': {'reject_as_good': {'keras (uncompressed)': {'deepdrid': 0.0855106888361045,
    'drimdb': 0.0,
    'fqs': 0.021321961620469083},
   'float32': {'deepdrid': 0.0855106888361045,
    'drimdb': 0.0,
    'fqs': 0.021321961620469083},
   'float16': {'deepdrid': 0.08432304038004751,
    'drimdb': 0.0,
    'fqs': 0.02025586353944563},
   'dynamic': {'deepdrid': 0.09144893111638955,
    'drimdb': 0.0,
    'fqs': 0.02025586353944563},
   'int8': {'deepdrid': 0.0332541567695962,
    'drimdb': 0.0,
    'fqs': 0.0021321961620469083}}},
 'notes': ''}

In [8]:
# ============================================================================
# CELL 8 - Does the PATHOLOGY control survive?
# ============================================================================
# Week 10 found the Reject rate rises with disease grade (+0.481) and showed the
# defect heads explain it. Quantisation could break that relationship without
# touching overall accuracy, so it gets re-measured per variant.
idr = ref[(ref.dataset == "idrid") & ref.dr_grade.notna()].copy()
print(f"{len(idr):,} IDRiD images with disease grades\n")
pos = ref.index.get_indexer(idr.index)
grades = idr.dr_grade.astype(int).values

def safe_spearman(a, b):
    """Spearman is undefined if either side is constant - say so, do not return NaN
    silently. A variant that predicts one class for every image would do this."""
    if len(np.unique(a)) < 2 or len(np.unique(b)) < 2:
        return float("nan")
    return float(spearmanr(a, b).statistic)


path_rows = []
r_ref = safe_spearman(grades, ref_grad[pos, REJECT])
d_ref = safe_spearman(grades, ref_sev[pos, DEFECTS.index("defocus")])
path_rows.append({"variant": "keras (uncompressed)", "grade_vs_p_reject": r_ref,
                  "grade_vs_defocus": d_ref,
                  "reject_rate": float((ref_grad.argmax(1)[pos] == REJECT).mean())})
for name in sizes:
    G, D = preds[name]
    path_rows.append({
        "variant": name,
        "grade_vs_p_reject": safe_spearman(grades, G[pos, REJECT]),
        "grade_vs_defocus": safe_spearman(grades, D[pos, DEFECTS.index("defocus")]),
        "reject_rate": float((G.argmax(1)[pos] == REJECT).mean())})

ptab = pd.DataFrame(path_rows).set_index("variant").round(3)
print(ptab.to_string())
print(f"""
  Both correlations should stay close to the uncompressed row ({r_ref:+.3f} and
  {d_ref:+.3f}). What matters is that they move TOGETHER: the defect head tracking
  disease grade at least as strongly as the reject probability is the evidence
  that the model is reporting real media opacity rather than reacting to lesions.
  A variant that keeps the first correlation but loses the second has kept the
  symptom and lost the explanation.

  A NaN here means that variant produced a constant value for every image, which
  is itself a failure - report it, do not average it away.""")

record("w12-pathology-under-compression",
       {"n_idrid": int(len(idr))},
       {"table": ptab.to_dict("index")})

516 IDRiD images with disease grades

                      grade_vs_p_reject  grade_vs_defocus  reject_rate
variant                                                               
keras (uncompressed)              0.481             0.516        0.062
float32                           0.481             0.516        0.062
float16                           0.482             0.515        0.060
dynamic                           0.507             0.550        0.054
int8                              0.338             0.167        0.190

  Both correlations should stay close to the uncompressed row (+0.481 and
  +0.516). What matters is that they move TOGETHER: the defect head tracking
  disease grade at least as strongly as the reject probability is the evidence
  that the model is reporting real media opacity rather than reacting to lesions.
  A variant that keeps the first correlation but loses the second has kept the
  symptom and lost the explanation.

  A NaN here means that variant prod

{'name': 'w12-pathology-under-compression',
 'when': '2026-10-13T22:31:09',
 'params': {'n_idrid': 516},
 'metrics': {'table': {'keras (uncompressed)': {'grade_vs_p_reject': 0.481,
    'grade_vs_defocus': 0.516,
    'reject_rate': 0.062},
   'float32': {'grade_vs_p_reject': 0.481,
    'grade_vs_defocus': 0.516,
    'reject_rate': 0.062},
   'float16': {'grade_vs_p_reject': 0.482,
    'grade_vs_defocus': 0.515,
    'reject_rate': 0.06},
   'dynamic': {'grade_vs_p_reject': 0.507,
    'grade_vs_defocus': 0.55,
    'reject_rate': 0.054},
   'int8': {'grade_vs_p_reject': 0.338,
    'grade_vs_defocus': 0.167,
    'reject_rate': 0.19}}},
 'notes': ''}

In [9]:
# ============================================================================
# CELL 9 - The decision
# ============================================================================
ref_rates = [v for k, v in rows[0].items() if k != "variant"]
assert ref_rates, ("no dataset has any human-labelled Reject images, so there is no "
                   "safety metric to preserve - check the manifest")
ref_worst = max(ref_rates)

dec = []
for name in sizes:
    a, b = agree[name], bench[name]
    worst_rag = max(safety[name].values()) if safety[name] else float("nan")
    dec.append({
        "variant": name,
        "size_MB": round(b["size_mb"], 2),
        "shrink": f"{KERAS_MB / b['size_mb']:.1f}x",
        "ms_img": round(b["ms_per_image"], 1),
        "confident_agreement": f"{a['argmax_agreement_confident']:.1%}",
        "defect_MAE": f"{a['defect_mae']:.1e}",
        "worst_reject_as_good": f"{worst_rag:.1%}",
        "vs_uncompressed": f"{worst_rag - ref_worst:+.1%}"})
dtab = pd.DataFrame(dec).set_index("variant")
print(f"original: {KERAS_MB:.2f} MB, worst reject-as-good {ref_worst:.1%}\n")
print(dtab.to_string())

print("""
HOW TO CHOOSE
  1. Rule out any variant whose worst reject-as-good rises meaningfully. Size is
     never worth that - this is a screening safety property, not a benchmark.
  2. Among what survives, prefer the one that is smaller AND faster. If float16
     or dynamic is slower than float32, it is buying file size with latency, and
     on a laptop waiting for one answer that is the wrong trade.
  3. Confident agreement below ~99% means the model genuinely behaves differently.
     Below ~95% it is a different model and should be re-validated from scratch,
     not shipped.

NOT RE-TESTED, AND WHY
  Grad-CAM localisation. It measured flat at the 0.43 chance level in both Week 10
  and Week 11, so compression has nothing to degrade. TFLite also has no autodiff,
  so the measurement is not constructible against a converted model. Stated here
  rather than quietly dropped.""")

record("w12-compression-decision",
       {"keras_mb": KERAS_MB, "reference_worst_reject_as_good": ref_worst},
       {"table": dtab.to_dict("index"), "bench": bench, "agreement": agree})

original: 18.37 MB, worst reject-as-good 8.6%

         size_MB shrink  ms_img confident_agreement defect_MAE worst_reject_as_good vs_uncompressed
variant                                                                                            
float32    17.36   1.1x    17.8              100.0%    1.5e-06                 8.6%           +0.0%
float16     8.74   2.1x    18.1              100.0%    1.0e-03                 8.4%           -0.1%
dynamic     4.88   3.8x    32.6               94.9%    2.7e-02                 9.1%           +0.6%
int8        5.25   3.5x    18.7               60.2%    1.0e-01                 3.3%           -5.2%

HOW TO CHOOSE
  1. Rule out any variant whose worst reject-as-good rises meaningfully. Size is
     never worth that - this is a screening safety property, not a benchmark.
  2. Among what survives, prefer the one that is smaller AND faster. If float16
     or dynamic is slower than float32, it is buying file size with latency, and
     on a laptop w

{'name': 'w12-compression-decision',
 'when': '2026-10-13T22:31:09',
 'params': {'keras_mb': 18.374802,
  'reference_worst_reject_as_good': 0.0855106888361045},
 'metrics': {'table': {'float32': {'size_MB': 17.36,
    'shrink': '1.1x',
    'ms_img': 17.8,
    'confident_agreement': '100.0%',
    'defect_MAE': '1.5e-06',
    'worst_reject_as_good': '8.6%',
    'vs_uncompressed': '+0.0%'},
   'float16': {'size_MB': 8.74,
    'shrink': '2.1x',
    'ms_img': 18.1,
    'confident_agreement': '100.0%',
    'defect_MAE': '1.0e-03',
    'worst_reject_as_good': '8.4%',
    'vs_uncompressed': '-0.1%'},
   'dynamic': {'size_MB': 4.88,
    'shrink': '3.8x',
    'ms_img': 32.6,
    'confident_agreement': '94.9%',
    'defect_MAE': '2.7e-02',
    'worst_reject_as_good': '9.1%',
    'vs_uncompressed': '+0.6%'},
   'int8': {'size_MB': 5.25,
    'shrink': '3.5x',
    'ms_img': 18.7,
    'confident_agreement': '60.2%',
    'defect_MAE': '1.0e-01',
    'worst_reject_as_good': '3.3%',
    'vs_uncompressed

In [10]:
# ============================================================================
# CELL 10 (CORRECTED) - Ship it: the deployable artefact + how to call it
# ============================================================================
# The previous tie-break sorted eligible variants on (latency, size). That put a
# 0.8 ms latency difference - 4.7% of an 18 ms budget - ahead of an 8.6 MB size
# difference, and selected a model that is barely compressed at all (1.1x).
#
# The deployment constraint is OFFLINE DISTRIBUTION TO FIELD LAPTOPS, so size is
# the thing being optimised and latency is a BUDGET, not a score: anything that
# answers one photograph well inside an operator's patience is equally acceptable.
# The rule below says that explicitly, and prints why each variant passed or failed
# so the decision is auditable rather than emergent.
LATENCY_BUDGET_MS = 200.0    # one eye photographed, one operator waiting
AGREEMENT_FLOOR   = 0.99     # below this it is a different model, not a smaller one
SAFETY_TOLERANCE  = 0.01     # reject-as-good may not rise by more than 1 point
PATHOLOGY_TOL     = 0.10     # both disease-grade correlations must be preserved

ref_pr = ptab.loc["keras (uncompressed)", "grade_vs_p_reject"]
ref_pd = ptab.loc["keras (uncompressed)", "grade_vs_defocus"]

print(f"reference: {KERAS_MB:.2f} MB | reject-as-good {ref_worst:.2%} | "
      f"pathology {ref_pr:+.3f} / {ref_pd:+.3f}\n")

eligible, why = [], {}
for name in sizes:
    a, b = agree[name], bench[name]
    worst = max(safety[name].values()) if safety[name] else float("inf")
    pr = ptab.loc[name, "grade_vs_p_reject"]
    pd_ = ptab.loc[name, "grade_vs_defocus"]
    fails = []
    if a["argmax_agreement_confident"] < AGREEMENT_FLOOR:
        fails.append(f"agreement {a['argmax_agreement_confident']:.1%}")
    if worst > ref_worst + SAFETY_TOLERANCE:
        fails.append(f"reject-as-good +{100 * (worst - ref_worst):.1f}pt")
    if abs(pr - ref_pr) > PATHOLOGY_TOL:
        fails.append(f"grade-vs-reject {pr:+.3f}")
    if abs(pd_ - ref_pd) > PATHOLOGY_TOL:
        fails.append(f"grade-vs-defocus {pd_:+.3f}")
    if b["ms_per_image"] > LATENCY_BUDGET_MS:
        fails.append(f"latency {b['ms_per_image']:.0f} ms")
    why[name] = fails
    if not fails:
        eligible.append(name)
    print(f"  {name:9s} {b['size_mb']:6.2f} MB  {b['ms_per_image']:5.1f} ms  "
          + ("ELIGIBLE" if not fails else "excluded: " + ", ".join(fails)))

assert eligible, "no variant met the deployment bar - ship the uncompressed model and say so"
CHOSEN = min(eligible, key=lambda n: bench[n]["size_mb"])     # smallest that qualifies
runner_up = sorted(eligible, key=lambda n: bench[n]["size_mb"])[1:2]

print(f"\nCHOSEN: {CHOSEN}  ->  {sizes[CHOSEN]:.2f} MB  "
      f"({KERAS_MB / sizes[CHOSEN]:.1f}x smaller), "
      f"{bench[CHOSEN]['ms_per_image']:.0f} ms/image, "
      f"{agree[CHOSEN]['argmax_agreement_confident']:.1%} confident agreement")
if runner_up:
    r = runner_up[0]
    print(f"  over {r} ({sizes[r]:.2f} MB, {bench[r]['ms_per_image']:.0f} ms): "
          f"{sizes[r] / sizes[CHOSEN]:.1f}x the size for "
          f"{100 * (bench[CHOSEN]['ms_per_image'] - bench[r]['ms_per_image']) / bench[r]['ms_per_image']:+.0f}% latency")

TEMPLATE = '''"""Offline inference - retina gradability + defect heads.
No network. No TensorFlow. Needs only: pip install tflite-runtime pillow numpy
"""
import numpy as np
from PIL import Image
try:
    from tflite_runtime.interpreter import Interpreter   # the ~2 MB runtime
except ImportError:
    from tensorflow.lite import Interpreter              # fallback

SIZE = __SIZE__
CLASSES = __CLASSES__
DEFECTS = __DEFECTS__

_it = Interpreter(model_path="retina_multihead___CHOSEN__.tflite")
_it.resize_tensor_input(_it.get_input_details()[0]["index"], [1, SIZE, SIZE, 3])
_it.allocate_tensors()
_in = _it.get_input_details()[0]["index"]

# Identify the two heads by SHAPE. TFLite output names are opaque and their order
# is not guaranteed, so reading them positionally would silently swap gradability
# for defects - and every shape downstream would still check out.
_by_units = {int(d["shape"][-1]): d["index"] for d in _it.get_output_details()}
_GRAD, _DEF = _by_units[3], _by_units[len(DEFECTS)]


def assess(path):
    """Returns (verdict, confidence, {defect: severity})."""
    img = Image.open(path).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR)
    x = np.asarray(img, dtype=np.float32)[None]   # [0, 255] - NOT normalised.
    _it.set_tensor(_in, x)                        # EfficientNet rescales internally.
    _it.invoke()
    g = _it.get_tensor(_GRAD)[0]
    d = _it.get_tensor(_DEF)[0]
    return CLASSES[int(g.argmax())], float(g.max()), dict(zip(DEFECTS, d.tolist()))


if __name__ == "__main__":
    import sys
    verdict, conf, defects = assess(sys.argv[1])
    print(f"{verdict}  ({conf:.0%} confident)")
    if verdict != "Good":
        worst = max(defects, key=defects.get)
        print(f"  most likely cause: {worst}  (severity {defects[worst]:.2f})")
'''
INFER = (TEMPLATE.replace("__SIZE__", str(SIZE))
                 .replace("__CLASSES__", repr(CLASSES))
                 .replace("__DEFECTS__", repr(DEFECTS))
                 .replace("__CHOSEN__", CHOSEN))
(WORK / "infer_offline.py").write_text(INFER)
print(f"\ninfer_offline.py written, pointing at retina_multihead_{CHOSEN}.tflite")

record("w12-compression-decision-final",
       {"latency_budget_ms": LATENCY_BUDGET_MS, "agreement_floor": AGREEMENT_FLOOR,
        "safety_tolerance": SAFETY_TOLERANCE, "pathology_tolerance": PATHOLOGY_TOL,
        "keras_mb": KERAS_MB},
       {"chosen": CHOSEN, "eligible": eligible, "exclusions": why,
        "chosen_size_mb": sizes[CHOSEN],
        "chosen_ms": bench[CHOSEN]["ms_per_image"],
        "chosen_agreement": agree[CHOSEN]["argmax_agreement_confident"]},
       notes="supersedes w12-compression-decision: the earlier tie-break ranked "
             "latency above size and selected a 1.1x-compressed model")

print("\n" + "=" * 72)
print("CORRECTED SUMMARY")
print("=" * 72)
print(dtab.to_string())
print(f"\nCHOSEN: {CHOSEN} ({sizes[CHOSEN]:.2f} MB) - supersedes the earlier float32 pick")
print("=" * 72)

reference: 18.37 MB | reject-as-good 8.55% | pathology +0.481 / +0.516

  float32    17.36 MB   17.8 ms  ELIGIBLE
  float16     8.74 MB   18.1 ms  ELIGIBLE
  dynamic     4.88 MB   32.6 ms  excluded: agreement 94.9%
  int8        5.25 MB   18.7 ms  excluded: agreement 60.2%, grade-vs-reject +0.338, grade-vs-defocus +0.167

CHOSEN: float16  ->  8.74 MB  (2.1x smaller), 18 ms/image, 100.0% confident agreement
  over float32 (17.36 MB, 18 ms): 2.0x the size for +2% latency

infer_offline.py written, pointing at retina_multihead_float16.tflite
recorded w12-compression-decision-final

CORRECTED SUMMARY
         size_MB shrink  ms_img confident_agreement defect_MAE worst_reject_as_good vs_uncompressed
variant                                                                                            
float32    17.36   1.1x    17.8              100.0%    1.5e-06                 8.6%           +0.0%
float16     8.74   2.1x    18.1              100.0%    1.0e-03                 8.4%           -0

---

## AFTER IT RUNS — outputs to save

### 1 · Publish as a new Kaggle Dataset

**Save Version → Save & Run All → Output tab → New Dataset**, named exactly:

```
retina-deployable
```

| File | Why it is kept |
|---|---|
| `tflite/retina_multihead_*.tflite` | All four variants. Keeping the ones you rejected is the evidence for the choice. |
| `infer_offline.py` | **The deliverable.** Loads the chosen model, identifies both heads by shape, and returns a verdict plus the most likely cause. No TensorFlow needed — `tflite-runtime` is about 2 MB. |
| `results/w12-compression-decision.json` | Size, latency, agreement and safety for every variant. |
| `results/w12-safety-under-compression.json` | Reject-as-Good per dataset per variant. |
| `results/w12-pathology-under-compression.json` | The disease-grade correlations per variant. |

### 2 · Paste the SUMMARY block back to me

**What I will be checking, in priority order:**

1. **`worst_reject_as_good` and `vs_uncompressed`.** If any variant raises the rate
   at which ungradable photographs are waved through, it is disqualified whatever it
   weighs. This is the one number the project exists for.
2. **`confident_agreement`.** Above 99% means the same model, smaller. Below 95% means
   a different model that would need re-validating.
3. **The latency column against size.** I expect float16 and dynamic to shrink the file
   while being no faster — possibly slower — than float32, and only int8 to improve
   both. If that pattern shows up, it is a genuine engineering finding worth a line in
   the report.
4. **`grade_vs_defocus` in the pathology table.** It must stay close to
   `grade_vs_p_reject`. If the reject correlation survives but the defocus one
   collapses, compression has kept the symptom and destroyed the explanation — which
   would undo the Week 10 result.

---

### If something goes wrong

| Symptom | Cause |
|---|---|
| Cell 1 `FileNotFoundError` | That dataset is not attached; the message names it |
| Cell 2 asserts on missing columns | `predictions_external.csv` is not Week 11's output |
| Cell 2 runs out of RAM | 4,556 × 224 × 224 × 3 float32 is ~2.7 GB. Decode in chunks, or sample `ref` down to 2,000 rows first |
| Cell 3 int8 conversion fails | Not fatal — the cell catches it and continues with the other three |
| Cell 4 asserts on output shapes | The model's head sizes changed; send me the printed shapes |
| Cell 6 is slow | Expected: 4,556 images × 4 variants at batch 1. That batch size is deliberate |

**Do not delete a failing assertion** — send me the output instead.